# The Sparlectra workshop tour

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Welthulk/Sparlectra.jl/blob/main/notebooks/workshop_tour.ipynb)

> **Note:** This workshop was created with AI assistance and is reviewed
> and curated by the maintainer; it is not a fully machine-generated text.

This tour is the FIRST HALF of the Sparlectra workshop: install
[Sparlectra.jl](https://github.com/Welthulk/Sparlectra.jl) once, warm the
compiler up once, and then climb from the very first bus to the solver's
control features in one session. The
[ADVANCED tour](https://colab.research.google.com/github/Welthulk/Sparlectra.jl/blob/main/notebooks/workshop_tour_advanced.ipynb)
continues with the Expert and Beyond tiers (remote voltage control,
HVDC, state estimation, FACTS, N-1, threads); the focused single-topic
notebooks go deeper on individual chapters.

After the warm-up (compilation happens there, everything after is fast)
the chapters climb three tiers:

**Newcomer**
1. Your first network, built step by step

**Beginner**
2. Working with the model: trust, switching, editing, Q-limits

**Advanced**
3. Slack types and short-circuit currents
4. Transformer tap control (OLTC)
5. Voltage-dependent reactive power, Q(U)

> **Note:** On Google Colab the install cell takes a few minutes on a
> fresh session (package download and precompilation). Colab's Julia
> version may change over time; this notebook targets Julia ≥ 1.12.

## Setup (Colab)
This cell installs Sparlectra from GitHub (branch `main`) into a fresh
temporary environment. The isolation matters on Colab: the shared
default environment ships many preinstalled packages, and installing
anything there triggers precompilation of that whole stack. Run this
cell first, once per session; it takes a few minutes.

In [1]:
using Pkg
Pkg.activate(temp = true)
Pkg.add(url = "https://github.com/Welthulk/Sparlectra.jl", rev = "main")
# To test another branch, set rev to its name, e.g. rev = "dev/r0.9.8".
# For the latest registered release use: Pkg.add("Sparlectra")
# Switching versions in a running session? A "[loaded: ...]" note means
# the old version is still active: restart the runtime, then rerun
# this cell.

  Activating new project at `/tmp/jl_YW1iKK`
     Cloning git-repo `https://github.com/Welthulk/Sparlectra.jl`
    Updating git-repo `https://github.com/Welthulk/Sparlectra.jl`
    Updating registry at `~/.julia/registries/General.toml`
   Resolving package versions...
   Installed InputBuffers ─── v1.1.1
   Installed XML2_jll ─────── v2.15.3+0
   Installed CodecInflate64 ─ v0.1.3
   Installed ZipArchives ──── v2.6.0
   Installed EzXML ────────── v1.2.3
   Installed BenchmarkTools ─ v1.8.0
   Installed ArgCheck ─────── v2.5.0
  Installing 1 artifacts
   Installed artifact XML2     2.2 MiB
    Updating `/tmp/jl_YW1iKK/Project.toml`
  [31ce9bba] + Sparlectra v0.9.16 `https://github.com/Welthulk/Sparlectra.jl#main`
    Updating `/tmp/jl_YW1iKK/Manifest.toml`
  [dce04be8] + ArgCheck v2.5.0
  [6e4b80f9] + BenchmarkTools v1.8.0
  [6309b1aa] + CodecInflate64 v0.1.3
  [944b1d66] + CodecZlib v0.7.9
  [34da2185] + Compat v4.18.1
  [8f5d6c58] + EzXML v1.2.3
  [0c81fc1b] + InputBuffers v1.1.1
  [6

## Warm-up and shared helpers

Julia compiles each function on first use. This one cell warms the
paths the chapters exercise, so nothing stalls mid-tour: the
Newton-Raphson solver and the IEC 60909 short circuit (chapter 3,
Example 3.4). The
`using` clauses and the small helpers of the whole tour live here too,
collected up top so they cannot be missed.

In [2]:
using Sparlectra
using Random

# solve helper used by all chapters (25 iterations, tolerance 1e-8)
function solve!(net; kwargs...)
  etime = @elapsed begin
    ite, erg = runpf!(net, 25, 1e-8, 0; kwargs...)
  end
  erg == 0 || error("Power flow did not converge (status = $erg)")
  calcNetLosses!(net)
  return etime, ite
end

# tiny warm-up net: a grid connection WITH declared short-circuit data
wnet = Net(name = "warmup", baseMVA = 100.0)
addBus!(net = wnet, busName = "A", vn_kV = 110.0)
addBus!(net = wnet, busName = "B", vn_kV = 110.0)
addExternalGrid!(net = wnet, busName = "A", vm_pu = 1.0, sk_max_MVA = 2000.0, sk_min_MVA = 1500.0, rx_max = 0.1, internal_impedance = false)
addProsumer!(net = wnet, busName = "B", type = "ENERGYCONSUMER", p = 10.0, q = 3.0)
addPIModelACLine!(net = wnet, fromBus = "A", toBus = "B", r_pu = 0.01, x_pu = 0.08, b_pu = 0.0, status = 1)

t_first = @elapsed runpf!(wnet, 10, 1e-8, 0)
t_second = @elapsed runpf!(wnet, 10, 1e-8, 0)
println("power flow     : first solve ", round(t_first; digits = 2), " s (compiles), second ", round(t_second * 1000; digits = 2), " ms")

t_sc = @elapsed runShortCircuit!(wnet; case = :max)
println("short circuit  : ", round(t_sc; digits = 2), " s, everything warm")

power flow     : first solve 32.97 s (compiles), second 0.36 ms
short circuit  : 2.36 s, everything warm


## Part I: Newcomer

## Chapter 1: your first network, built step by step

**Example 1.1: a 7-bus ring, built and solved.** No input files, no
configuration: a complete 110 kV network from scratch, validated,
solved, and read. The network is seven buses in a ring with
two cross-connections; `B1` carries the grid connection:

```text
 (slack)
   B1 ---- B2 ---- B3 ---- B4
   |         \    /         |
   |          \  /          |
   |           \/           |     diagonals: B2-B5 and B3-B6
   |           /\           |
   B7 ---- B6 ---- B5 ------+
```

Every Sparlectra model starts from a `Net` object; `baseMVA` is the
system base power for all per-unit conversions. `addBus!` creates the
electrical nodes (`vn_kV` nominal voltage, `vm_pu`/`va_deg` the solver's
starting voltage). Note what is NOT declared here: the operational bus
type (slack / PV / PQ) is derived later from the devices attached to
each bus.

In [3]:
net1 = Net(name = "tour_first_pf", baseMVA = 100.0)
addBus!(net = net1, busName = "B1", vn_kV = 110.0, vm_pu = 1.02, va_deg = 0.0)
for i in 2:7
  addBus!(net = net1, busName = "B$(i)", vn_kV = 110.0, vm_pu = 1.0, va_deg = 0.0)
end

`addPIModelACLine!` connects buses with a line as a pi-equivalent branch
in per-unit (`r_pu`, `x_pu`, and `b_pu` for the total charging):

In [4]:
addPIModelACLine!(net = net1, fromBus = "B1", toBus = "B2", r_pu = 0.010, x_pu = 0.080, b_pu = 0.0, status = 1)
addPIModelACLine!(net = net1, fromBus = "B2", toBus = "B3", r_pu = 0.011, x_pu = 0.085, b_pu = 0.0, status = 1)
addPIModelACLine!(net = net1, fromBus = "B3", toBus = "B4", r_pu = 0.012, x_pu = 0.090, b_pu = 0.0, status = 1)
addPIModelACLine!(net = net1, fromBus = "B4", toBus = "B5", r_pu = 0.010, x_pu = 0.080, b_pu = 0.0, status = 1)
addPIModelACLine!(net = net1, fromBus = "B5", toBus = "B6", r_pu = 0.011, x_pu = 0.085, b_pu = 0.0, status = 1)
addPIModelACLine!(net = net1, fromBus = "B6", toBus = "B7", r_pu = 0.012, x_pu = 0.090, b_pu = 0.0, status = 1)
addPIModelACLine!(net = net1, fromBus = "B7", toBus = "B1", r_pu = 0.010, x_pu = 0.080, b_pu = 0.0, status = 1)
addPIModelACLine!(net = net1, fromBus = "B2", toBus = "B5", r_pu = 0.009, x_pu = 0.070, b_pu = 0.0, status = 1)
addPIModelACLine!(net = net1, fromBus = "B3", toBus = "B6", r_pu = 0.009, x_pu = 0.070, b_pu = 0.0, status = 1)

9-element Vector{Branch}:
 Branch( Component(ID=#B_ACL_110_1_2#1, Name=B_ACL_110_1_2, Typ=BranchC, Vn=110.0, From_bus=1, To_bus=2, ), branchIdx: 1, fromBus: 1, toBus: 2, r_pu: 0.01, x_pu: 0.08, b_pu: 0.0, g_pu: 0.0, ratio: 0.0, angle: 0.0, status: 1, tap_ratio: 1.0, phase_shift_deg: 0.0, )

 Branch( Component(ID=#B_ACL_110_2_3#2, Name=B_ACL_110_2_3, Typ=BranchC, Vn=110.0, From_bus=2, To_bus=3, ), branchIdx: 2, fromBus: 2, toBus: 3, r_pu: 0.011, x_pu: 0.085, b_pu: 0.0, g_pu: 0.0, ratio: 0.0, angle: 0.0, status: 1, tap_ratio: 1.0, phase_shift_deg: 0.0, )

 Branch( Component(ID=#B_ACL_110_3_4#3, Name=B_ACL_110_3_4, Typ=BranchC, Vn=110.0, From_bus=3, To_bus=4, ), branchIdx: 3, fromBus: 3, toBus: 4, r_pu: 0.012, x_pu: 0.09, b_pu: 0.0, g_pu: 0.0, ratio: 0.0, angle: 0.0, status: 1, tap_ratio: 1.0, phase_shift_deg: 0.0, )

 Branch( Component(ID=#B_ACL_110_4_5#4, Name=B_ACL_110_4_5, Typ=BranchC, Vn=110.0, From_bus=4, To_bus=5, ), branchIdx: 4, fromBus: 4, toBus: 5, r_pu: 0.01, x_pu: 0.08, b_pu:

Devices that consume or produce power are `addProsumer!` calls. The
external network injection at `B1` references its OWN bus as the voltage
reference; that is what makes `B1` the slack bus. The generator at `B3`
feeds in 60 MW, the remaining buses carry loads (`p` in MW, `q` in MVar):

In [5]:
addProsumer!(net = net1, busName = "B1", type = "EXTERNALNETWORKINJECTION", referencePri = "B1", vm_pu = 1.02, va_deg = 0.0)
addProsumer!(net = net1, busName = "B3", type = "GENERATOR", p = 60.0, q = 10.0)
addProsumer!(net = net1, busName = "B2", type = "LOAD", p = 35.0, q = 10.0)
addProsumer!(net = net1, busName = "B4", type = "LOAD", p = 45.0, q = 15.0)
addProsumer!(net = net1, busName = "B5", type = "LOAD", p = 25.0, q = 8.0)
addProsumer!(net = net1, busName = "B6", type = "LOAD", p = 30.0, q = 10.0)
addProsumer!(net = net1, busName = "B7", type = "LOAD", p = 20.0, q = 6.0)

`validate!` checks the model for structural problems (unconnected buses,
missing slack, inconsistent parameters) BEFORE any numerics run; make it
a habit after every round of model edits. Then `runpf!` runs the
rectangular Newton-Raphson solver (max iterations, tolerance, verbosity;
status `0` means converged), `calcNetLosses!` derives branch flows and
losses from the converged voltages, and `printACPFlowResults` prints the
classical result tables:

In [6]:
ok1, msg1 = validate!(net = net1)
ok1 || error("Network validation failed: $msg1")
etime, ite = solve!(net1)   ## solve! wraps exactly runpf! + calcNetLosses! (see warm-up)
printACPFlowResults(net1, etime, ite, 1e-8)

| SPARLECTRA Version 0.9.16     - AC Power Flow Results                        |
Date           :   24-Aug-26 8:20:42
Iterations     :         4
Flatstart      :        No
Tolerance      : 1.0e-08
Solver         :             NR
Total time     : 0.019259 s
Case           :  tour_first_pf
Cooldown iters :         0
Q-hysteresis   :    0.0000 pu
Jacobian cond. : kappa1(J) = 45.0, attainable accuracy ~ 1.0e-14, well conditioned (tol 1.0e-8 reachable)
BaseMVA        :       100
Nodes          :         7 (PV: 0 PQ: 6 Slack: 1)
Grid connection: slack bus B1
Branches       :         9
Links          :         0
HVDC links     :         0
Lines          :         9
Trafos         :         0
Generators     :         2
Loads          :         5
Shunts         :         0
Controllers    :         0 (Tap: 0, Q(U): 0, P(U): 0)
PV→PQ locks    :         0
PV→PQ events   :         0

total network power balance (Σ S_branch): P =      0.923 [MW], Q =      7.242 [MVar]

| Nr    | Bus                 

Reading aid (Example 1.1): the slack at `B1` covers the difference
between 155 MW of
load, 60 MW of scheduled generation, and the network losses; all bus
voltages stay near 1.0 pu. Loading a case from a FILE instead is one
call through the framework workflow:
`run_sparlectra(casefile = "case14.m", path = ...)` after
`ensure_casefile("case14.m")`, which downloads the case on demand; the
result carries the solved net as `result.net`.

The same construction, packed into a function: later examples reuse
this network (model editing in Examples 2.3 and 2.4, state estimation
in the advanced tour).

In [7]:
function build_ring7(name::String)
  net = Net(name = name, baseMVA = 100.0)
  addBus!(net = net, busName = "B1", vn_kV = 110.0, vm_pu = 1.02, va_deg = 0.0)
  for i in 2:7
    addBus!(net = net, busName = "B$(i)", vn_kV = 110.0, vm_pu = 1.0, va_deg = 0.0)
  end
  addPIModelACLine!(net = net, fromBus = "B1", toBus = "B2", r_pu = 0.010, x_pu = 0.080, b_pu = 0.0, status = 1)
  addPIModelACLine!(net = net, fromBus = "B2", toBus = "B3", r_pu = 0.011, x_pu = 0.085, b_pu = 0.0, status = 1)
  addPIModelACLine!(net = net, fromBus = "B3", toBus = "B4", r_pu = 0.012, x_pu = 0.090, b_pu = 0.0, status = 1)
  addPIModelACLine!(net = net, fromBus = "B4", toBus = "B5", r_pu = 0.010, x_pu = 0.080, b_pu = 0.0, status = 1)
  addPIModelACLine!(net = net, fromBus = "B5", toBus = "B6", r_pu = 0.011, x_pu = 0.085, b_pu = 0.0, status = 1)
  addPIModelACLine!(net = net, fromBus = "B6", toBus = "B7", r_pu = 0.012, x_pu = 0.090, b_pu = 0.0, status = 1)
  addPIModelACLine!(net = net, fromBus = "B7", toBus = "B1", r_pu = 0.010, x_pu = 0.080, b_pu = 0.0, status = 1)
  addPIModelACLine!(net = net, fromBus = "B2", toBus = "B5", r_pu = 0.009, x_pu = 0.070, b_pu = 0.0, status = 1)
  addPIModelACLine!(net = net, fromBus = "B3", toBus = "B6", r_pu = 0.009, x_pu = 0.070, b_pu = 0.0, status = 1)
  addProsumer!(net = net, busName = "B1", type = "EXTERNALNETWORKINJECTION", referencePri = "B1", vm_pu = 1.02, va_deg = 0.0)
  addProsumer!(net = net, busName = "B3", type = "GENERATOR", p = 60.0, q = 10.0)
  addProsumer!(net = net, busName = "B2", type = "LOAD", p = 35.0, q = 10.0)
  addProsumer!(net = net, busName = "B4", type = "LOAD", p = 45.0, q = 15.0)
  addProsumer!(net = net, busName = "B5", type = "LOAD", p = 25.0, q = 8.0)
  addProsumer!(net = net, busName = "B6", type = "LOAD", p = 30.0, q = 10.0)
  addProsumer!(net = net, busName = "B7", type = "LOAD", p = 20.0, q = 6.0)
  ok, msg = validate!(net = net)
  ok || error("Network validation failed: $msg")
  return net
end

build_ring7 (generic function with 1 method)

## Part II: Beginner

## Chapter 2: working with the model

Solving once is the easy part. This chapter covers what day-to-day work
actually consists of: judging how much the numbers can be trusted,
editing and switching the model, exporting it, and letting the solver
enforce reactive-power limits.

### How much can you trust these numbers?

Every Newton iteration solves the linear system $J \, \Delta x = -F$ with
the power-flow Jacobian $J$. The condition number $\kappa(J)$ measures how
strongly that solve amplifies tiny perturbations: rounding, measurement
noise in the input data, small parameter changes. The attainable relative
accuracy in Float64 is roughly $\kappa \cdot 2 \cdot 10^{-16}$, so every
power of ten in $\kappa$ costs one significant digit of the result.
`condestJacobian(net)` estimates $\kappa_1$ at the operating point the net
currently holds, on the same sparse Jacobian the solver factors.
**Example 2.1: the condition number of the healthy ring.** First the
solved 7-bus ring of Example 1.1 (diagram there):

In [8]:
println("ring network: kappa = ", round(condestJacobian(net1), sigdigits = 3))

ring network: kappa = 45.0


Reading aid (Example 2.1): around 45, excellent. Rule of thumb: below
about $10^6$ well conditioned, around $10^{10}$ borderline, beyond
$10^{14}$ numerically singular in Float64.

The instructive part is how conditioning degrades when the physics
degenerate, long before the solver visibly fails. **Example 2.2: a
feeder with a degenerating stub.** Take a small feeder with a
measurement stub at `B3` and make the stub line weaker in each round:

```text
 (slack)
   B1 -------- B2 - - - - B3     stub line B2-B3: x_pu grows
             20 MW               from 0.08 to 8e10 per round
```

In [9]:
for x_weak in (0.08, 800.0, 8.0e6, 8.0e10)
  net = Net(name = "tour_cond", baseMVA = 100.0)
  addBus!(net = net, busName = "B1", vn_kV = 110.0)
  addBus!(net = net, busName = "B2", vn_kV = 110.0)
  addBus!(net = net, busName = "B3", vn_kV = 110.0)
  addProsumer!(net = net, busName = "B1", type = "EXTERNALNETWORKINJECTION", referencePri = "B1", vm_pu = 1.0, va_deg = 0.0)
  addProsumer!(net = net, busName = "B2", type = "ENERGYCONSUMER", p = 20.0, q = 5.0)
  addPIModelACLine!(net = net, fromBus = "B1", toBus = "B2", r_pu = 0.01, x_pu = 0.08, b_pu = 0.0, status = 1)
  addPIModelACLine!(net = net, fromBus = "B2", toBus = "B3", r_pu = x_weak / 8, x_pu = x_weak, b_pu = 0.0, status = 1)
  _, ite_weak = solve!(net)
  vm3 = round(something(net.nodeVec[3]._vm_pu, NaN); digits = 4)
  println("x = ", lpad(x_weak, 8), " pu:  ", ite_weak, " iterations,  Vm(B3) = ", vm3, ",  kappa = ", round(condestJacobian(net), sigdigits = 3))
end

x =     0.08 pu:  4 iterations,  Vm(B3) = 0.9938,  kappa = 11.6
x =    800.0 pu:  4 iterations,  Vm(B3) = 0.9938,  kappa = 13000.0
x =    8.0e6 pu:  4 iterations,  Vm(B3) = 0.9938,  kappa = 1.3e8
x =   8.0e10 pu:  4 iterations,  Vm(B3) = 0.9938,  kappa = 1.3e12


Reading aid (Example 2.2): the solver converges in 4 iterations in
every round, and
`Vm(B3)` prints the same plausible 0.9938 each time. Nothing in the result
table reveals that the last round sits at $\kappa \approx 10^{12}$, where
only about 4 significant digits survive: the printed voltage is already at
the edge of what the arithmetic can guarantee, and any sensitivity built
on this Jacobian (voltage per tap step, voltage per MVar) is numerically
meaningless. That is exactly what the estimate is for: the classic result
log reports it as a `Jacobian cond.` line, and diagnose runs grade it with
a plain-language verdict.

### Editing, switching, exporting

**Example 2.3: an edit round on the ring.** Model work is iterative:
change a parameter, switch an element, remove one, validate, solve
again; the stage is a fresh copy of the 7-bus ring of Example 1.1
(diagram there). The dedicated helpers keep the bookkeeping
consistent (branch indices, prosumer injections, isolated buses):

In [26]:
net_edit = build_ring7("tour_edit")
# stiffen the B1-B2 line (per-branch parameter update)
brVec = getNetBranchNumberVec(net = net_edit, fromBus = "B1", toBus = "B2")
updateBranchParameters!(net = net_edit, branchNr = brVec[1], branch = BranchModel(0.005, 0.040, 0.0, 0.0, 0.0, 0.0, 100.0))
# add 5 MW / 1 MVAr of load at B4. Loads and generators are PROSUMER
# objects and the AC solver reads its injections from them, so growing a
# load means adding (or editing) a prosumer. The node-sum helpers
# (addBusLoadPower!) only feed the report layer, not the AC solve (#323).
addProsumer!(net = net_edit, busName = "B4", type = "LOAD", p = 5.0, q = 1.0)
# switch the B3-B6 cross-tie out of service (aggregate switch; a single
# terminal would be setBranchTerminalStatus!, see Example 2.6)
tie = getNetBranchNumberVec(net = net_edit, fromBus = "B3", toBus = "B6")
setNetBranchStatus!(net = net_edit, branchNr = tie[1], status = 0)
# remove the B2-B5 cross-tie outright, then re-validate and solve
removeACLine!(net = net_edit, fromBus = "B2", toBus = "B5")
markIsolatedBuses!(net = net_edit, log = false)
ok_e, msg_e = validate!(net = net_edit)
ok_e || error("edit round left the net invalid: $msg_e")
etime, ite_e = solve!(net_edit)
println("edited net solves in ", ite_e, " iterations; branches now: ", length(net_edit.branchVec))
# the edited model exports as a MATPOWER case file
case_out = joinpath(mktempdir(), "tour_edit.m")
writeMatpowerCasefile(net_edit, case_out)
println("exported: ", case_out, " (", filesize(case_out), " bytes)")
printACPFlowResults(net_edit, etime, ite_e, 1e-8)

edited net solves in 5 iterations; branches now: 8
exported: /tmp/jl_lzGE9Y/tour_edit.m (3348 bytes)
| SPARLECTRA Version 0.9.16     - AC Power Flow Results                        |
Date           :    24-Aug-26 8:45:0
Iterations     :         5
Flatstart      :        No
Tolerance      : 1.0e-08
Solver         :             NR
Total time     : 0.002050 s
Case           :      tour_edit
Cooldown iters :         0
Q-hysteresis   :    0.0000 pu
Jacobian cond. : kappa1(J) = 32.6, attainable accuracy ~ 7.2e-15, well conditioned (tol 1.0e-8 reachable)
BaseMVA        :       100
Nodes          :         7 (PV: 0 PQ: 6 Slack: 1)
Grid connection: slack bus B1
Branches       :         8
Links          :         0
HVDC links     :         0
Lines          :         8
Trafos         :         0
Generators     :         2
Loads          :         6
Shunts         :         0
Controllers    :         0 (Tap: 0, Q(U): 0, P(U): 0)
PV→PQ locks    :         0
PV→PQ events   :         0

total network p

[ Info: convertion to Matpower CASE-Files, Testcase: (tour_edit)


Reading aid (Example 2.3): removal helpers (`removeACLine!`, `removeTrafo!`,
`removeProsumer!`, ...) mutate the net and can leave isolated buses
behind; `markIsolatedBuses!` flags them for the solver and
`clearIsolatedBuses!` deletes the safe ones. `removeBus!` deliberately
only CHECKS removability. Re-validate after every edit round.

**Example 2.4: the node-level power trap.** One trap worth demonstrating
(issue #323), still on the edited ring of Example 2.3: there are also
NODE-level power helpers (`addBusLoadPower!`, `addBusGenPower!`). They
edit report sums that NO solver reads; the solvers build their
injections from the prosumer objects. So this "edit" changes nothing:

In [11]:
vm_before = get_bus_vm_pu(net_edit, "B4")
addBusLoadPower!(net = net_edit, busName = "B4", p = 25.0, q = 5.0)  ## report layer only!
solve!(net_edit)
println("after addBusLoadPower!(+25 MW): Vm(B4) = ", round(get_bus_vm_pu(net_edit, "B4"); digits = 4), " pu (before: ", round(vm_before; digits = 4), " pu, unchanged)")
addProsumer!(net = net_edit, busName = "B4", type = "LOAD", p = 25.0, q = 5.0)  ## THIS is a load
solve!(net_edit)
println("after addProsumer!(LOAD, 25 MW): Vm(B4) = ", round(get_bus_vm_pu(net_edit, "B4"); digits = 4), " pu (sags, the solver saw it)")

after addBusLoadPower!(+25 MW): Vm(B4) = 0.9706 pu (before: 0.9706 pu, unchanged)
after addProsumer!(LOAD, 25 MW): Vm(B4) = 0.9556 pu (sags, the solver saw it)


### Reactive-power limits (PV to PQ switching)

A voltage-regulating machine holds its bus voltage only while its
reactive power stays inside `[qMin, qMax]`. When the solver hits a
limit, it switches the bus from PV to PQ at the violated bound within
the Newton iteration (the active-set strategy) and reports every event.
**Example 2.5: a machine pinned at its Q-limit.** A three-bus chain, the
regulating machine in the middle:

```text
 (slack)
   Q1 -------- Q2 -------- Q3
          gen 20 MW      load 45 MW / 20 MVAr
          1.05 pu, Q in
          [-5, 5] MVAr
```

In [12]:
net_q = Net(name = "tour_qlimits", baseMVA = 100.0)
for b in ("Q1", "Q2", "Q3")
  addBus!(net = net_q, busName = b, vn_kV = 110.0)
end
addProsumer!(net = net_q, busName = "Q1", type = "EXTERNALNETWORKINJECTION", referencePri = "Q1", vm_pu = 1.0, va_deg = 0.0)
addProsumer!(net = net_q, busName = "Q2", type = "GENERATOR", p = 20.0, vm_pu = 1.05, qMin = -5.0, qMax = 5.0)
addProsumer!(net = net_q, busName = "Q3", type = "ENERGYCONSUMER", p = 45.0, q = 20.0)
addPIModelACLine!(net = net_q, fromBus = "Q1", toBus = "Q2", r_pu = 0.01, x_pu = 0.08, b_pu = 0.0, status = 1)
addPIModelACLine!(net = net_q, fromBus = "Q2", toBus = "Q3", r_pu = 0.01, x_pu = 0.08, b_pu = 0.0, status = 1)
validate!(net = net_q)
solve!(net_q)
println("Vm(Q2) = ", round(get_bus_vm_pu(net_q, "Q2"); digits = 4), " pu (setpoint was 1.05)")
printQLimitLog(net_q)
distributeBusResults!(net_q)

Vm(Q2) = 0.9833 pu (setpoint was 1.05)
PV->PQ switching events:
  total events     : 1
  rows shown       : 1
  rows omitted     : 0
──────────────────────────────────
 Iteration │ Bus │ Side
──────────────────────────────────
         2 │   2 │ max 
──────────────────────────────────


Reading aid (Example 2.5): holding 1.05 pu at `Q2` would need more
than the 5 MVar the
machine may deliver, so the solver pins Q at `qMax` and lets the voltage
float below the setpoint; the log names bus, iteration, and bound.
`distributeBusResults!` pushes the solved bus totals back onto the
individual prosumers; with several machines on one bus it redistributes
water-filling style, so no unit leaves its Q range. The deep dive
(enforcement modes, guards, oscillation handling) is on the
[Q-limit strategy page](https://welthulk.github.io/Sparlectra.jl/q_limit_switching_strategy/).

### Opening one end of a line

A breaker can open a single terminal while the other stays connected.
The line then carries no through flow, but it does NOT disappear: seen
from the closed bus it collapses to its exact pi reduction and keeps
drawing its FULL charging (for realistic lines the two shunt arms act
almost in parallel, so it is `g + jb`, not half of it), plus the small
ohmic loss of the charging current. The voltage at the open end rises
above the feeding bus, the classical Ferranti effect, and is reported as
a branch result without adding a bus. **Example 2.6: the Ferranti rise,
and why "fully disconnected" is the wrong model for it.** Three states of
the same network: line closed, the breaker at the B end open, and the
line treated as completely disconnected. The second corridor to `C` keeps
the system solvable in every state, so the states are comparable:

```text
         (slack)
  C ------- A =============== B     A=B: one long 380-kV line, b_pu = 0.9;
  50 MW           120 MW           the breaker at the B end opens
```

In [13]:
# Example 2.6, state 1 (closed): the long line feeds the 120-MW load
net_open = Net(name = "tour_open_end", baseMVA = 100.0)
for b in ("A", "B", "C")
  addBus!(net = net_open, busName = b, vn_kV = 380.0)
end
addProsumer!(net = net_open, busName = "A", type = "EXTERNALNETWORKINJECTION", referencePri = "A", vm_pu = 1.0, va_deg = 0.0)
addProsumer!(net = net_open, busName = "B", type = "ENERGYCONSUMER", p = 120.0, q = 30.0)
addProsumer!(net = net_open, busName = "C", type = "ENERGYCONSUMER", p = 50.0, q = 10.0)
addPIModelACLine!(net = net_open, fromBus = "A", toBus = "B", r_pu = 0.02, x_pu = 0.16, b_pu = 0.9, g_pu = 0.004, status = 1)
addPIModelACLine!(net = net_open, fromBus = "A", toBus = "C", r_pu = 0.01, x_pu = 0.08, b_pu = 0.02, status = 1)
validate!(net = net_open)
solve!(net_open)
println("state 1, closed : line A=B carries ", round(get_branch_p_from_to_mw(net_open, "A", "B"); digits = 1), " MW to the load")

# Example 2.6, state 2 (open@to): open the breaker at the B end and
# re-solve; the classical result tables below show the consequences
setBranchTerminalStatus!(net_open.branchVec[1]; to = false)
markIsolatedBuses!(net = net_open, log = false)
etime_open, ite_open = solve!(net_open)
br_open = net_open.branchVec[1]
q_slack_open = get_branch_q_from_to_mvar(net_open, "A", "B") + get_branch_q_from_to_mvar(net_open, "A", "C")
println("state 2, open@to: charging draw ", round(br_open.fBranchFlow.qFlow; digits = 1), " MVAr, active loss ", round(br_open.fBranchFlow.pFlow; digits = 3), " MW")
println("         voltage at the OPEN end: ", round(br_open.open_end_vm_pu; digits = 4), " pu, HIGHER than the ", round(get_bus_vm_pu(net_open, "A"); digits = 2), " pu at the feeding bus A")
println("         (Ferranti effect: the charging current flowing through the line reactance lifts the voltage toward the open end)")

# the classical result print of state 2: the branch row carries the
# open@to marker, the header counts one open terminal, and the isolated
# bus B shows the OPEN-END voltage in its V columns (the Ferranti value,
# flagged open-end in the Control column)
printACPFlowResults(net_open, etime_open, ite_open, 1e-8)

# Example 2.6, state 3 (fully disconnected): the WRONG model for an open
# breaker end; the pi stub vanishes from the Y-bus and with it the
# charging draw and the Ferranti information
setBranchTerminalStatus!(net_open.branchVec[1]; from = false)
markIsolatedBuses!(net = net_open, log = false)
solve!(net_open)
q_slack_off = get_branch_q_from_to_mvar(net_open, "A", "C")
println("state 3, fully disconnected: open-end voltage ", br_open.open_end_vm_pu === nothing ? "gone" : "?", ", slack reactive supply now ", round(q_slack_off; digits = 1), " MVAr")
println("reactive balance shift state 2 -> state 3: ", round(q_slack_off - q_slack_open; digits = 1), " MVAr of charging draw silently vanished")

state 1, closed : line A=B carries 123.5 MW to the load
state 2, open@to: charging draw -93.5 MVAr, active loss 0.902 MW
         voltage at the OPEN end: 1.0775 pu, HIGHER than the 1.0 pu at the feeding bus A
         (Ferranti effect: the charging current flowing through the line reactance lifts the voltage toward the open end)
| SPARLECTRA Version 0.9.16     - AC Power Flow Results                        |
Date           :   24-Aug-26 8:20:51
Iterations     :         1
Flatstart      :        No
Tolerance      : 1.0e-08
Solver         :             NR
Total time     : 0.000283 s
Case           :  tour_open_end
Cooldown iters :         0
Q-hysteresis   :    0.0000 pu
Jacobian cond. : kappa1(J) = 1.42, attainable accuracy ~ 3.2e-16, well conditioned (tol 1.0e-8 reachable)
BaseMVA        :       100
Nodes          :         3 (PV: 0 PQ: 2 Slack: 1)
Grid connection: slack bus A
Branches       :         2
Open terminals :         1 (branches open at one terminal)
  open end     : B_ACL_3

Reading aid (Example 2.6): in state 2 the classical tables carry the
whole story: the branch row is marked `open@to` with the open-end
voltage, the header counts it under `Open terminals`, and the isolated
bus `B` shows the Ferranti voltage (1.0775 pu) in its V columns, flagged
`open-end` because it is a branch RESULT (`open_end_vm_pu`) measured at
the open breaker, not a solved bus voltage. State 3 is the modeling trap
the feature exists for: treating the one-sided opening as a full
disconnect hides roughly 93 MVAr of charging draw from the reactive
balance and erases the Ferranti overvoltage a protection engineer would
care about. The full story, including the Schur reduction and why it is
the full charging, is on the branch-model docs page under "One-sided open
branches"; the runnable twin is `exp_open_terminal_line.jl`.

### Links: connections without impedance

A link (`addLink!`) models a busbar coupler or sectionalizer: a closed
switch between two buses. It is NOT a branch. It has no impedance, it is
never stamped into the Y-bus, and it never appears in the branch table.
Instead the solver contracts every cluster of buses joined by closed
links onto one representative bus before the Y-bus is built, so all
linked buses share one voltage by construction. (Do not confuse these
links with the HVDC "Link" rows of the advanced tour's chapter 2: a bus
link is a switch, an HVDC link is a converter pair.)

Because the link has no admittance, the power flow cannot tell how much
power crosses it: that is reconstructed AFTER the solve, from Kirchhoff's
current law, with `calcLinkFlowsKCL!`. **Example 2.7: a zero-impedance
ring of links.** The interesting case is a ring of links, a
zero-impedance cycle:

```text
     S (slack)
     |
     |  real line (r = 0.01, x = 0.08)
     |
     R1
    /  \            R1, R2, R3 joined by three closed links:
   /    \           an impedance-less ring, electrically ONE node
  R3 --- R2
(20 MW) (30 MW)
```

In a zero-impedance loop the flow split is physically NOT unique: any
circulating current can be added without changing a single voltage.
Sparlectra returns the minimum-norm KCL solution (Moore-Penrose
pseudoinverse), the unique split with zero artificial circulation, so
the result is deterministic and reproducible:

In [27]:
net_ring = Net(name = "tour_link_ring", baseMVA = 100.0)
addBus!(net = net_ring, busName = "S", vn_kV = 110.0)
for b in ("R1", "R2", "R3")
  addBus!(net = net_ring, busName = b, vn_kV = 110.0)
end
addProsumer!(net = net_ring, busName = "S", type = "EXTERNALNETWORKINJECTION", referencePri = "S", vm_pu = 1.0, va_deg = 0.0)
addProsumer!(net = net_ring, busName = "R2", type = "ENERGYCONSUMER", p = 30.0, q = 8.0)
addProsumer!(net = net_ring, busName = "R3", type = "ENERGYCONSUMER", p = 20.0, q = 5.0)
addPIModelACLine!(net = net_ring, fromBus = "S", toBus = "R1", r_pu = 0.01, x_pu = 0.08, b_pu = 0.0, status = 1)
addLink!(net = net_ring, fromBus = "R1", toBus = "R2", status = 1)
addLink!(net = net_ring, fromBus = "R2", toBus = "R3", status = 1)
addLink!(net = net_ring, fromBus = "R3", toBus = "R1", status = 1)
validate!(net = net_ring)
etime, ite = solve!(net_ring)
calcLinkFlowsKCL!(net_ring)

println("one voltage for the whole ring: Vm(R1/R2/R3) = ", join((round(get_bus_vm_pu(net_ring, b); digits = 5) for b in ("R1", "R2", "R3")), " / "), " pu")
ring_bus_name = Dict(v => k for (k, v) in net_ring.busDict)   ## BusLink stores bus INDICES
for l in net_ring.linkVec
  println("link ", ring_bus_name[l.fromBus], " -> ", ring_bus_name[l.toBus], ": ", lpad(round(l.pFlow_MW; digits = 2), 7), " MW")
end
printACPFlowResults(net_ring, etime, ite, 1e-8)

one voltage for the whole ring: Vm(R1/R2/R3) = 0.98357 / 0.98357 / 0.98357 pu
link R1 -> R2:   26.67 MW
link R2 -> R3:   -3.33 MW
link R3 -> R1:  -23.33 MW
| SPARLECTRA Version 0.9.16     - AC Power Flow Results                        |
Date           :   24-Aug-26 8:47:39
Iterations     :         4
Flatstart      :        No
Tolerance      : 1.0e-08
Solver         :             NR
Total time     : 0.002467 s
Case           : tour_link_ring
Cooldown iters :         0
Q-hysteresis   :    0.0000 pu
Jacobian cond. : kappa1(J) = 1.42, attainable accuracy ~ 3.2e-16, well conditioned (tol 1.0e-8 reachable)
BaseMVA        :       100
Nodes          :         4 (PV: 0 PQ: 1 Slack: 1)
Grid connection: slack bus S
PF Nodes       :         2 (after active-link merge)
Branches       :         1
Links          :         3
HVDC links     :         0
Lines          :         1
Trafos         :         0
Generators     :         1
Loads          :         2
Shunts         :         0
Controllers    : 

Reading aid (Example 2.7): 50 MW enter the ring at `R1`. The
minimum-norm split sends
26.67 MW directly to the 30 MW load at `R2` and 23.33 MW the other way
round to `R3`; the third coupler carries only the 3.33 MW that `R2` still
needs. A negative sign just means the flow runs against the link's
from-to direction. Any other split (say 30 and 20 with an idle third
coupler) would satisfy KCL too, but only by adding a circulating
component; the pseudoinverse is exactly the split without one. The
links page of the docs has the math and the modeling guidelines (for
example: never link the slack bus itself).

## Part III: Advanced

## Chapter 3: slack types and short-circuit currents

One grid connection, modeled three ways, plus an IEC 60909-0 fault-current
sweep from the declared feeder data. The detailed walk-through with full
reading aids is the
[slack-types notebook](https://colab.research.google.com/github/Welthulk/Sparlectra.jl/blob/main/notebooks/workshop_slack_short_circuit.ipynb).

All examples of this chapter share one 8-bus network (`build_grid`):
a ring with two chords, generators at `B3` and `B6`, loads at `B2`,
`B4`, `B7`, and `B8`, and at `B1` the grid connection whose model the
examples vary:

```text
 (grid)                G
   B1 ------ B2 ------ B3 ------ B4
   |          |         |         |     ring of eight buses plus
   |          |         |         |     the chords B2-B7 and B3-B6
   B8 ------ B7 ------ B6 ------ B5
                        G
```

In [15]:
function build_grid(mode::Symbol)
  net = Net(name = "tour_eg8_$(mode)", baseMVA = 100.0)
  for b in ("B1", "B2", "B3", "B4", "B5", "B6", "B7", "B8")
    addBus!(net = net, busName = b, vn_kV = 110.0)
  end
  addPIModelACLine!(net = net, fromBus = "B1", toBus = "B2", r_pu = 0.010, x_pu = 0.060, b_pu = 0.02, status = 1)
  addPIModelACLine!(net = net, fromBus = "B2", toBus = "B3", r_pu = 0.015, x_pu = 0.080, b_pu = 0.02, status = 1)
  addPIModelACLine!(net = net, fromBus = "B3", toBus = "B4", r_pu = 0.020, x_pu = 0.090, b_pu = 0.02, status = 1)
  addPIModelACLine!(net = net, fromBus = "B4", toBus = "B5", r_pu = 0.012, x_pu = 0.070, b_pu = 0.02, status = 1)
  addPIModelACLine!(net = net, fromBus = "B5", toBus = "B6", r_pu = 0.015, x_pu = 0.075, b_pu = 0.02, status = 1)
  addPIModelACLine!(net = net, fromBus = "B6", toBus = "B7", r_pu = 0.018, x_pu = 0.085, b_pu = 0.02, status = 1)
  addPIModelACLine!(net = net, fromBus = "B7", toBus = "B8", r_pu = 0.010, x_pu = 0.055, b_pu = 0.02, status = 1)
  addPIModelACLine!(net = net, fromBus = "B8", toBus = "B1", r_pu = 0.011, x_pu = 0.065, b_pu = 0.02, status = 1)
  addPIModelACLine!(net = net, fromBus = "B2", toBus = "B7", r_pu = 0.020, x_pu = 0.100, b_pu = 0.02, status = 1)
  addPIModelACLine!(net = net, fromBus = "B3", toBus = "B6", r_pu = 0.022, x_pu = 0.110, b_pu = 0.02, status = 1)
  addProsumer!(net = net, busName = "B3", type = "GENERATOR", p = 60.0, vm_pu = 1.01, qMin = -60.0, qMax = 60.0)
  addProsumer!(net = net, busName = "B6", type = "GENERATOR", p = 40.0, vm_pu = 1.00, qMin = -40.0, qMax = 40.0)
  addProsumer!(net = net, busName = "B2", type = "ENERGYCONSUMER", p = 45.0, q = 12.0)
  addProsumer!(net = net, busName = "B4", type = "ENERGYCONSUMER", p = 50.0, q = 15.0)
  addProsumer!(net = net, busName = "B7", type = "ENERGYCONSUMER", p = 40.0, q = 10.0)
  addProsumer!(net = net, busName = "B8", type = "ENERGYCONSUMER", p = 25.0, q = 8.0)
  addExternalGrid!(net = net, busName = "B1", vm_pu = 1.02, sk_max_MVA = 2000.0, sk_min_MVA = 1500.0, rx_max = 0.1, internal_impedance = (mode === :source))
  ok, msg = validate!(net = net)
  ok || error("Network validation failed: $msg")
  return net
end

build_grid (generic function with 1 method)

**Example 3.1: the ideal slack.** `B1` is pinned at exactly
1.02 pu / 0° and absorbs the whole imbalance (the `SLACK` row).

In [16]:
net_slack = build_grid(:slack)
etime, ite = solve!(net_slack)
printACPFlowResults(net_slack, etime, ite, 1e-8)

| SPARLECTRA Version 0.9.16     - AC Power Flow Results                        |
Date           :   24-Aug-26 8:20:54
Iterations     :         4
Flatstart      :        No
Tolerance      : 1.0e-08
Solver         :             NR
Total time     : 0.000771 s
Case           : tour_eg8_slack
Cooldown iters :         0
Q-hysteresis   :    0.0000 pu
Jacobian cond. : kappa1(J) = 310.0, attainable accuracy ~ 6.9e-14, well conditioned (tol 1.0e-8 reachable)
BaseMVA        :       100
Nodes          :         8 (PV: 2 PQ: 5 Slack: 1)
Grid connection: slack bus B1
Branches       :        10
Links          :         0
HVDC links     :         0
Lines          :        10
Trafos         :         0
Generators     :         3
Loads          :         4
Shunts         :         0
Controllers    :         0 (Tap: 0, Q(U): 0, P(U): 0)
PV→PQ locks    :         0
PV→PQ events   :         0

total network power balance (Σ S_branch): P =      0.821 [MW], Q =    -15.938 [MVar]

| Nr    | Bus                

**Example 3.2: the non-ideal source.** With `internal_impedance = true`
the setpoint moves to the hidden internal bus (last row, type `SOURCE`);
the terminal `B1` in the first row droops below 1.02 pu.

In [17]:
net_source = build_grid(:source)
etime, ite = solve!(net_source)
printACPFlowResults(net_source, etime, ite, 1e-8)

| SPARLECTRA Version 0.9.16     - AC Power Flow Results                        |
Date           :   24-Aug-26 8:20:54
Iterations     :         4
Flatstart      :        No
Tolerance      : 1.0e-08
Solver         :             NR
Total time     : 0.000742 s
Case           :tour_eg8_source
Cooldown iters :         0
Q-hysteresis   :    0.0000 pu
Jacobian cond. : kappa1(J) = 378.0, attainable accuracy ~ 8.4e-14, well conditioned (tol 1.0e-8 reachable)
BaseMVA        :       100
Nodes          :         9 (PV: 2 PQ: 6 Slack: 0 Source: 1)
Grid connection: external-grid source at B1 (Sk'' = 2000.0 MVA, R/X = 0.1; internal slack: B1__extgrid_int)
Branches       :        11
Links          :         0
HVDC links     :         0
Lines          :        11
Trafos         :         0
Generators     :         3
Loads          :         4
Shunts         :         0
Controllers    :         0 (Tap: 0, Q(U): 0, P(U): 0)
PV→PQ locks    :         0
PV→PQ events   :         0

total network power balance

**Example 3.3: distributed slack.** The generators pick up the imbalance
according to their scheduled output (0.6/0.4); the slack row keeps only
the reactive balance.

In [18]:
net_dist = build_grid(:slack)
etime, ite = solve!(net_dist; distributed_slack_enabled = true, distributed_slack_p_mode = :pg_weighted)
printACPFlowResults(net_dist, etime, ite, 1e-8)

| SPARLECTRA Version 0.9.16     - AC Power Flow Results                        |
Date           :   24-Aug-26 8:20:56
Iterations     :         4
Flatstart      :        No
Tolerance      : 1.0e-08
Solver         :             NR
Total time     : 1.436966 s
Case           : tour_eg8_slack
Cooldown iters :         0
Q-hysteresis   :    0.0000 pu
Jacobian cond. : kappa1(J) = 306.0, attainable accuracy ~ 6.8e-14, well conditioned (tol 1.0e-8 reachable)
BaseMVA        :       100
Nodes          :         8 (PV: 2 PQ: 5 Slack: 1)
Grid connection: slack bus B1
Branches       :        10
Links          :         0
HVDC links     :         0
Lines          :        10
Trafos         :         0
Generators     :         3
Loads          :         4
Shunts         :         0
Controllers    :         0 (Tap: 0, Q(U): 0, P(U): 0)
PV→PQ locks    :         0
PV→PQ events   :         0
Dist. slack    : mode pg_weighted, lambda_P = +61.537 MW (imbalance + losses picked up by 2 participant(s), see the 

The three connection models of Examples 3.1 to 3.3 side by side. The
losses differ because the flow pattern differs; a negative Q loss means
the line charging produces more reactive power than the flows consume.

In [19]:
println(rpad("scenario", 20), lpad("Vm(B1) pu", 11), lpad("P loss MW", 11), lpad("Q loss MVAr", 13), "   balanced by")
for (label, net, by) in (
  ("ideal slack", net_slack, "slack bus B1"),
  ("non-ideal source", net_source, "hidden source bus"),
  ("distributed slack", net_dist, "B3 (α=0.6) + B6 (α=0.4)"),
)
  pl, ql = getTotalLosses(net = net)
  println(rpad(label, 20), lpad(string(round(get_bus_vm_pu(net, "B1"); digits = 4)), 11), lpad(string(round(pl; digits = 3)), 11), lpad(string(round(ql; digits = 3)), 13), "   ", by)
end

scenario              Vm(B1) pu  P loss MW  Q loss MVAr   balanced by
ideal slack                1.02      0.821      -15.938   slack bus B1
non-ideal source         1.0087      0.974      -14.086   hidden source bus
distributed slack          1.02      1.537      -12.415   B3 (α=0.6) + B6 (α=0.4)


**Example 3.4: the IEC 60909 fault sweep.** The feeder data declared in
`addExternalGrid!` feeds `runShortCircuit!` directly, here on the
ideal-slack net of Example 3.1. $I_k''$ is largest at the connection bus
and decays with electrical distance.

In [20]:
printShortCircuitResult(runShortCircuit!(net_slack; case = :max))
printShortCircuitResult(runShortCircuit!(net_slack; case = :min))

Balanced 3-phase short circuit (IEC 60909-0) — case: max, c per IEC Table 1
bus                     Un[kV]   Ik''[kA]   Sk''[MVA]   kappa   ip[kA]     status     flagged
B1                      110.0    10.497     2000.0      2.0     29.691     ok         no
B2                      110.0    5.703      1086.6      1.931   15.574     ok         no
B3                      110.0    3.8549     734.47      1.8862  10.283     ok         no
B4                      110.0    2.9546     562.94      1.8565  7.7573     ok         no
B5                      110.0    2.978      567.39      1.8597  7.8324     ok         no
B6                      110.0    3.6897     702.97      1.8749  9.7832     ok         no
B7                      110.0    4.8744     928.69      1.908   13.153     ok         no
B8                      110.0    5.5627     1059.8      1.9266  15.156     ok         no
Balanced 3-phase short circuit (IEC 60909-0) — case: min, c per IEC Table 1
bus                     Un[kV]   Ik''[kA] 

## Chapter 4: transformer tap control (OLTC)

**Example 4.1: an OLTC holding a remote bus.** A transformer with a
ratio tap changer holds the voltage at a remote load bus. The outer
control loop moves the discrete tap until the target is inside the
deadband; the power flow itself stays untouched. Details:
[Control Framework](https://welthulk.github.io/Sparlectra.jl/control_framework/).

```text
 (slack)
   Slack ==T1== MV -------- Load       T1: ratio taps 0.9..1.1,
                       60 MW / 20 MVAr     step 0.0125
```

In [21]:
function build_oltc()
  net = Net(name = "tour_oltc", baseMVA = 100.0)
  addBus!(net = net, busName = "Slack", vn_kV = 110.0)
  addBus!(net = net, busName = "MV", vn_kV = 110.0)
  addBus!(net = net, busName = "Load", vn_kV = 110.0)
  addProsumer!(net = net, busName = "Slack", type = "EXTERNALNETWORKINJECTION", referencePri = "Slack", vm_pu = 1.0, va_deg = 0.0)
  addProsumer!(net = net, busName = "Load", type = "ENERGYCONSUMER", p = 60.0, q = 20.0)
  addPIModelTrafo!(net = net, fromBus = "Slack", toBus = "MV", r_pu = 0.004, x_pu = 0.06, b_pu = 0.0, ratio = 1.0, shift_deg = 0.0, status = 1)
  addPIModelACLine!(net = net, fromBus = "MV", toBus = "Load", r_pu = 0.02, x_pu = 0.10, b_pu = 0.01, status = 1)
  # enable the ratio tap on the transformer branch and give it a name the
  # controller can address
  t = getNetBranch(net = net, fromBus = "Slack", toBus = "MV")
  t.comp.cName = "T1"
  t.has_ratio_tap = true
  t.tap_min = 0.90
  t.tap_max = 1.10
  t.tap_step = 0.0125
  ok, msg = validate!(net = net)
  ok || error("Network validation failed: $msg")
  return net
end

net_oltc = build_oltc()
run_sparlectra(net = net_oltc)
println("uncontrolled: Vm(Load) = ", round(get_bus_vm_pu(net_oltc, "Load"); digits = 4), " pu")

uncontrolled: Vm(Load) = 0.9474 pu


Now attach the controller (voltage mode, discrete steps) and rerun.
`run_sparlectra` executes the outer control loop automatically when
controllers are present.

In [22]:
addTapController!(
  net_oltc;
  trafo = "T1",
  mode = :voltage,
  target_bus = "Load",
  target_vm_pu = 1.0,
  control_ratio = true,
  control_phase = false,
  is_discrete = true,
  deadband_vm_pu = 5e-3,
)
run_sparlectra(net = net_oltc)
println("controlled:   Vm(Load) = ", round(get_bus_vm_pu(net_oltc, "Load"); digits = 4), " pu")
printTapControllerSummary(stdout, net_oltc)

controlled:   Vm(Load) = 1.0036 pu

Transformer Control Summary
---------------------------
Power sign convention: achieved_p_mw is positive in the configured target branch direction (from -> to).
T1 OLTC (T1, 1 -> 2)
  controller type    : OLTC
  mode               : voltage
  target bus         : Load
  target Vm          : 1.0000 pu
  achieved Vm        : 1.0036 pu
  target branch      : -
  target P           : -
  achieved P         : -
  tap ratio          : 0.95000
  phase shift        : 0.00000 deg
  tap position       : -4
  phase position     : +0
  ratio range        : 0.90000 .. 1.10000
  ratio step         : 0.01250
  phase range        : -30.00000 .. 30.00000 deg
  phase step         : 1.25000 deg
  discrete           : true
  converged          : true
  at_limit           : false
  status             : converged


Reading aid (Example 4.1): the summary shows the chosen tap position
and the achieved
voltage. With a discrete 0.0125 step the controller stops as soon as the
target is inside the deadband, not at the exact setpoint.

## Chapter 5: voltage-dependent reactive power, Q(U)

A machine can follow a Q(U) droop characteristic: absorb reactive power
when its voltage is high, inject when it is low. Unlike the outer-loop
controllers above, Q(U) is solved **inside** Newton-Raphson. Details:
[Voltage Dependent Control](https://welthulk.github.io/Sparlectra.jl/voltage_dependent_control/).

Both examples of this chapter run one three-bus feeder with the Q(U)
machine in the middle; only the load at `B3` changes:

```text
 (slack, 1.02 pu)
   B1 -------- B2 -------- B3
          Q(U) machine   load (light in Example 5.1,
          10 MW          heavy in Example 5.2)
```

In [23]:
function build_qu(p_load::Float64, q_load::Float64)
  net = Net(name = "tour_qu", baseMVA = 100.0)
  addBus!(net = net, busName = "B1", vn_kV = 110.0)
  addBus!(net = net, busName = "B2", vn_kV = 110.0)
  addBus!(net = net, busName = "B3", vn_kV = 110.0)
  addProsumer!(net = net, busName = "B1", type = "EXTERNALNETWORKINJECTION", referencePri = "B1", vm_pu = 1.02, va_deg = 0.0)
  addPIModelACLine!(net = net, fromBus = "B1", toBus = "B2", r_pu = 0.01, x_pu = 0.08, b_pu = 0.0, status = 1)
  addPIModelACLine!(net = net, fromBus = "B2", toBus = "B3", r_pu = 0.01, x_pu = 0.08, b_pu = 0.0, status = 1)
  qu = QUController(
    make_characteristic(
      [(104.5, 30.0), (107.0, 20.0), (110.0, 0.0), (112.0, -10.0), (115.5, -20.0)];
      voltage_unit = :kV,
      value_unit = :MVAr,
      vn_kV = 110.0,
      sbase_MVA = 100.0,
      interpolation = :polynomial,
    );
    qmin_MVAr = -50.0,
    qmax_MVAr = 50.0,
    sbase_MVA = 100.0,
  )
  addProsumer!(net = net, busName = "B2", type = "SYNCHRONOUSMACHINE", p = 10.0, q = 0.0, qu_controller = qu)
  addProsumer!(net = net, busName = "B3", type = "ENERGYCONSUMER", p = p_load, q = q_load)
  ok, msg = validate!(net = net)
  ok || error("Network validation failed: $msg")
  return net
end

build_qu (generic function with 1 method)

**Example 5.1: light load.** The machine bus sits above 110 kV, so the
characteristic asks the machine to **absorb** reactive power (negative Q).

In [24]:
net_qu_light = build_qu(5.0, 1.0)
etime, ite = solve!(net_qu_light)
printACPFlowResults(net_qu_light, etime, ite, 1e-8)

| SPARLECTRA Version 0.9.16     - AC Power Flow Results                        |
Date           :   24-Aug-26 8:21:20
Iterations     :         4
Flatstart      :        No
Tolerance      : 1.0e-08
Solver         :             NR
Total time     : 0.000448 s
Case           :        tour_qu
Cooldown iters :         0
Q-hysteresis   :    0.0000 pu
Jacobian cond. : kappa1(J) = 11.2, attainable accuracy ~ 2.5e-15, well conditioned (tol 1.0e-8 reachable)
BaseMVA        :       100
Nodes          :         3 (PV: 0 PQ: 2 Slack: 1)
Grid connection: slack bus B1
Branches       :         2
Links          :         0
HVDC links     :         0
Lines          :         2
Trafos         :         0
Generators     :         2
Loads          :         1
Shunts         :         0
Controllers    :         1 (Tap: 0, Q(U): 1, P(U): 0)
PV→PQ locks    :         0
PV→PQ events   :         0

total network power balance (Σ S_branch): P =      0.013 [MW], Q =      0.101 [MVar]

| Nr    | Bus                 

**Example 5.2: heavy load.** The voltage sags below 110 kV and the same
characteristic turns the machine into a reactive power **injector**
(positive Q).

In [25]:
net_qu_heavy = build_qu(80.0, 25.0)
etime, ite = solve!(net_qu_heavy)
printACPFlowResults(net_qu_heavy, etime, ite, 1e-8)

| SPARLECTRA Version 0.9.16     - AC Power Flow Results                        |
Date           :   24-Aug-26 8:21:20
Iterations     :         5
Flatstart      :        No
Tolerance      : 1.0e-08
Solver         :             NR
Total time     : 0.000605 s
Case           :        tour_qu
Cooldown iters :         0
Q-hysteresis   :    0.0000 pu
Jacobian cond. : kappa1(J) = 14.2, attainable accuracy ~ 3.2e-15, well conditioned (tol 1.0e-8 reachable)
BaseMVA        :       100
Nodes          :         3 (PV: 0 PQ: 2 Slack: 1)
Grid connection: slack bus B1
Branches       :         2
Links          :         0
HVDC links     :         0
Lines          :         2
Trafos         :         0
Generators     :         2
Loads          :         1
Shunts         :         0
Controllers    :         1 (Tap: 0, Q(U): 1, P(U): 0)
PV→PQ locks    :         0
PV→PQ events   :         0

total network power balance (Σ S_branch): P =      1.335 [MW], Q =     10.682 [MVar]

| Nr    | Bus                 

Reading aid (Examples 5.1 and 5.2): compare the `Qg` value and the
`Control` column (`Q(U)`) of bus `B2` between the two tables; the sign
flips with the voltage level, exactly along the declared characteristic.

## Where to go next

The workshop continues in the
[ADVANCED tour](https://colab.research.google.com/github/Welthulk/Sparlectra.jl/blob/main/notebooks/workshop_tour_advanced.ipynb)
(Expert and Beyond tiers): remote voltage control by a machine, a
steerable HVDC link, state estimation, the FACTS devices and their
limits, N-1 contingency analysis, and parallel sweeps on Julia threads.

The focused notebooks with the full narrative of single topics:

- [Slack types and short circuit](https://colab.research.google.com/github/Welthulk/Sparlectra.jl/blob/main/notebooks/workshop_slack_short_circuit.ipynb)
- [Distributed slack](https://colab.research.google.com/github/Welthulk/Sparlectra.jl/blob/main/notebooks/workshop_distributed_slack.ipynb)
- [Transformer taps](https://colab.research.google.com/github/Welthulk/Sparlectra.jl/blob/main/notebooks/workshop_transformers.ipynb)
- [TCSC flow steering](https://colab.research.google.com/github/Welthulk/Sparlectra.jl/blob/main/notebooks/workshop_series_compensation.ipynb)
- [State estimation](https://colab.research.google.com/github/Welthulk/Sparlectra.jl/blob/main/notebooks/workshop_state_estimation.ipynb)

And the documentation for going further:

- [Solver Guide](https://welthulk.github.io/Sparlectra.jl/solver/)
- [Slack and External Grid Sources](https://welthulk.github.io/Sparlectra.jl/slack_vs_source/)
- [Control Framework](https://welthulk.github.io/Sparlectra.jl/control_framework/)
- [Voltage Dependent Control](https://welthulk.github.io/Sparlectra.jl/voltage_dependent_control/)
- [Feature Matrix](https://welthulk.github.io/Sparlectra.jl/feature_matrix/)